# DriveValue AI: Model Optimization & Explainability

## Capstone Project

**Author:** Jacqualine Makgolana

## Purpose

The purpose of this notebook is to improve the predictive performance of the selected machine learning model and increase its interpretability.

The Random Forest model identified in Notebook 4 will be optimized using hyperparameter tuning. In addition, Explainable AI techniques will be used to understand how different vehicle characteristics influence predicted asking prices.

These improvements enhance both the accuracy and transparency of the DriveValue AI pricing system.

## Business Objective

A machine learning model should not only produce accurate predictions but also provide explanations that stakeholders can understand and trust.

This notebook aims to optimize model performance while identifying the vehicle characteristics that have the greatest influence on price predictions.

## Success Criteria

The optimized model should:

- Improve prediction accuracy over the baseline Random Forest model.
- Reduce prediction error (MAE and RMSE).
- Increase the R² score.
- Produce interpretable feature explanations.
- Support trustworthy pricing recommendations for dealerships and online marketplaces.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import plotly.express as px

from sklearn.ensemble import RandomForestRegressor

from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [ ]:
X_train = pd.read_csv("../Data/processed/X_train.csv")
X_test = pd.read_csv("../Data/processed/X_test.csv")

y_train = pd.read_csv("../Data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("../Data/processed/y_test.csv").squeeze()

print("Training Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)

Training Shape: (11990, 466)
Testing Shape : (2998, 466)


## Dataset Overview

The processed training and testing datasets created during the preprocessing stage have been loaded successfully.

These datasets will be used to optimize the Random Forest model identified as the best-performing algorithm in Notebook 4.

## Hyperparameter Optimization (RandomizedSearchCV)

## Hyperparameter Optimization

The Random Forest model achieved the best predictive performance in Notebook 4. However, its performance can often be improved by carefully selecting optimal hyperparameter values.

Rather than relying on default settings, Randomized Search Cross Validation is used to evaluate multiple combinations of hyperparameters and identify the configuration that produces the best predictive performance.

The optimized model will then be compared against the original Random Forest model.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

rf_base = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)

random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring="r2",
    verbose=2,
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)

print("Hyperparameter tuning completed!")

Fitting 5 folds for each of 20 candidates, totalling 100 fits
Hyperparameter tuning completed!


In [ ]:
print("Best Parameters:\n")

random_search.best_params_

Best Parameters:



{'n_estimators': 300,
 'min_samples_split': 5,
 'min_samples_leaf': 1,
 'max_features': 'sqrt',
 'max_depth': None}

In [ ]:
best_rf = random_search.best_estimator_

best_predictions = best_rf.predict(X_test)

best_mae = mean_absolute_error(y_test, best_predictions)

best_rmse = np.sqrt(
    mean_squared_error(y_test, best_predictions)
)

best_r2 = r2_score(
    y_test,
    best_predictions
)

print(f"MAE : {best_mae:,.2f}")
print(f"RMSE: {best_rmse:,.2f}")
print(f"R²  : {best_r2:.4f}")

MAE : 408,006.99
RMSE: 1,287,550.53
R²  : 0.4284


In [ ]:
optimization_results = pd.DataFrame({
    "Model": [
        "Original Random Forest",
        "Optimized Random Forest"
    ],
    "MAE": [
        398156.267135,
        best_mae
    ],
    "RMSE": [
        1267200.992500,
        best_rmse
    ],
    "R2": [
        0.446325,
        best_r2
    ]
})

optimization_results

,Model,MAE,RMSE,R2
0,Original Random Forest,398156.267135,1.267201e+06,0.446325
1,Optimized Random Forest,408006.992743,1.287551e+06,0.428399


## overview

Hyperparameter tuning was performed using RandomizedSearchCV. Although several parameter combinations were evaluated, the optimized model did not outperform the original Random Forest on the unseen test data. Therefore, the original Random Forest model was retained as the final production model because it achieved the highest R² score (0.446) and the lowest prediction errors."

## Explainable AI (SHAP)

# Explainable AI (SHAP)

While predictive accuracy is important, understanding *why* a machine learning model makes its predictions is equally valuable.

Random Forest is considered a "black-box" model because it combines hundreds of decision trees, making individual predictions difficult to interpret.

To improve transparency, SHAP (SHapley Additive exPlanations) is used to quantify the contribution of each feature to the model's predictions. This provides interpretable insights into how different vehicle characteristics influence asking prices.

Explainable AI increases trust in the model and supports informed decision-making for dealerships, online marketplaces, and vehicle sellers.

In [ ]:
import shap

print("SHAP imported successfully!")

SHAP imported successfully!


In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Random Forest model loaded successfully!")

Random Forest model loaded successfully!


In [ ]:
xrf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("Random Forest model trained successfully!")

Random Forest model trained successfully!


In [ ]:
print(rf)

RandomForestRegressor(n_jobs=-1, random_state=42)


## SHAP Value Computation

SHAP (SHapley Additive exPlanations) assigns an importance value to every feature for every prediction generated by the Random Forest model.

Unlike traditional feature importance, SHAP explains not only which features matter but also whether each feature increases or decreases the predicted vehicle price.

This provides local and global interpretability, making the DriveValue AI model transparent and trustworthy for business decision-making.

In [ ]:
explainer = shap.TreeExplainer(rf)

shap_values = explainer.shap_values(X_test)

print("SHAP values calculated successfully!")